# Music Recommendation Systems Comparison

This notebook demonstrates and compares different types of recommendation systems using our music dataset:

1. **Content-Based Filtering**
2. **Collaborative Filtering** (with simulated user data)
3. **Hybrid Approach**
4. **Performance Comparison**

---

## 1. Data Loading and Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import NMF
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

In [ ]:
# Load the music dataset
music_df = pd.read_csv('music_dataset.csv')

print("📊 Music Dataset Overview")
print("=" * 50)
print(f"Total entries: {len(music_df)}")
print(f"Unique artists: {music_df['name'].nunique()}")
print(f"Unique genres: {music_df['genre_list'].nunique()}")
print("\nFirst 10 entries:")
print(music_df.head(10))

# Basic statistics
print("\n📈 Genre Distribution (Top 10):")
genre_counts = music_df['genre_list'].value_counts().head(10)
print(genre_counts)

In [ ]:
# Visualize genre distribution
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
genre_counts.plot(kind='bar')
plt.title('Top 10 Most Common Genres')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45)

plt.subplot(1, 2, 2)
artist_genre_counts = music_df.groupby('name').size().sort_values(ascending=False).head(10)
artist_genre_counts.plot(kind='bar')
plt.title('Artists with Most Genres')
plt.xlabel('Artist')
plt.ylabel('Number of Genres')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

---
## 2. Content-Based Filtering System

Content-based filtering recommends items based on the features/characteristics of the items themselves.

In [ ]:
class ContentBasedRecommender:
    def __init__(self, music_df):
        self.music_df = music_df
        self.artist_genres = self._create_artist_genre_matrix()
        self.similarity_matrix = self._calculate_similarity()
        
    def _create_artist_genre_matrix(self):
        """Create a matrix where each artist has their genres as a concatenated string"""
        artist_genres = self.music_df.groupby('name')['genre_list'].apply(lambda x: ' '.join(x)).reset_index()
        return artist_genres
    
    def _calculate_similarity(self):
        """Calculate cosine similarity between artists based on their genres"""
        # Use TF-IDF to vectorize the genre strings
        tfidf = TfidfVectorizer()
        tfidf_matrix = tfidf.fit_transform(self.artist_genres['genre_list'])
        
        # Calculate cosine similarity
        similarity_matrix = cosine_similarity(tfidf_matrix)
        return similarity_matrix
    
    def recommend(self, artist_name, n_recommendations=5):
        """Recommend similar artists based on content similarity"""
        if artist_name not in self.artist_genres['name'].values:
            return f"Artist '{artist_name}' not found in dataset"
        
        # Get the index of the artist
        artist_idx = self.artist_genres[self.artist_genres['name'] == artist_name].index[0]
        
        # Get similarity scores for this artist
        sim_scores = list(enumerate(self.similarity_matrix[artist_idx]))
        
        # Sort by similarity score (excluding the artist itself)
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n_recommendations+1]
        
        # Get recommended artists
        recommendations = []
        for idx, score in sim_scores:
            artist = self.artist_genres.iloc[idx]['name']
            genres = self.artist_genres.iloc[idx]['genre_list']
            recommendations.append({
                'artist': artist,
                'similarity_score': score,
                'genres': genres
            })
        
        return recommendations
    
    def get_artist_genres(self, artist_name):
        """Get genres for a specific artist"""
        artist_data = self.artist_genres[self.artist_genres['name'] == artist_name]
        if len(artist_data) > 0:
            return artist_data['genre_list'].iloc[0]
        return "Artist not found"

In [ ]:
# Initialize and test Content-Based Recommender
content_recommender = ContentBasedRecommender(music_df)

# Test with different artists
test_artists = ['Taylor Swift', 'The Beatles', 'Drake', 'Miles Davis']

print("🎵 CONTENT-BASED RECOMMENDATIONS")
print("=" * 60)

for artist in test_artists:
    print(f"\n🎤 Artist: {artist}")
    print(f"Genres: {content_recommender.get_artist_genres(artist)}")
    print("\nRecommendations:")
    
    recommendations = content_recommender.recommend(artist, n_recommendations=3)
    
    if isinstance(recommendations, str):
        print(recommendations)
    else:
        for i, rec in enumerate(recommendations, 1):
            print(f"{i}. {rec['artist']} (Similarity: {rec['similarity_score']:.3f})")
            print(f"   Genres: {rec['genres']}")
    
    print("-" * 50)

---
## 3. Collaborative Filtering System

Since we don't have user rating data, let's simulate user-artist ratings to demonstrate collaborative filtering.

In [ ]:
# Create simulated user-artist rating data
np.random.seed(42)

# Get unique artists
unique_artists = music_df['name'].unique()
n_users = 100
n_artists = len(unique_artists)

# Create user IDs
user_ids = [f"User_{i+1}" for i in range(n_users)]

# Simulate ratings (1-5 scale) with some sparsity
ratings_data = []

for user_id in user_ids:
    # Each user rates only some artists (create sparsity)
    n_ratings = np.random.randint(10, 30)  # Each user rates 10-30 artists
    rated_artists = np.random.choice(unique_artists, n_ratings, replace=False)
    
    for artist in rated_artists:
        # Generate ratings with some bias towards certain genres
        artist_genres = music_df[music_df['name'] == artist]['genre_list'].tolist()
        
        # Base rating
        rating = np.random.randint(1, 6)
        
        # Add some genre preferences (simulated user preferences)
        if 'Pop' in artist_genres:
            rating += np.random.choice([-1, 0, 1], p=[0.2, 0.6, 0.2])
        if 'Rock' in artist_genres:
            rating += np.random.choice([-1, 0, 1], p=[0.1, 0.7, 0.2])
        
        rating = max(1, min(5, rating))  # Ensure rating is between 1-5
        
        ratings_data.append({
            'user_id': user_id,
            'artist': artist,
            'rating': rating
        })

# Create ratings DataFrame
ratings_df = pd.DataFrame(ratings_data)

print("📊 Simulated User Ratings Dataset")
print("=" * 40)
print(f"Total ratings: {len(ratings_df)}")
print(f"Users: {ratings_df['user_id'].nunique()}")
print(f"Artists rated: {ratings_df['artist'].nunique()}")
print(f"Average rating: {ratings_df['rating'].mean():.2f}")
print(f"Rating distribution:")
print(ratings_df['rating'].value_counts().sort_index())

print("\nSample ratings:")
print(ratings_df.head(10))

In [ ]:
# Create user-artist rating matrix
user_artist_matrix = ratings_df.pivot(index='user_id', columns='artist', values='rating').fillna(0)

print(f"User-Artist Matrix Shape: {user_artist_matrix.shape}")
print(f"Sparsity: {(user_artist_matrix == 0).sum().sum() / (user_artist_matrix.shape[0] * user_artist_matrix.shape[1]) * 100:.1f}%")

# Display a sample of the matrix
print("\nSample of User-Artist Rating Matrix:")
print(user_artist_matrix.iloc[:5, :5])

In [ ]:
class CollaborativeFilteringRecommender:
    def __init__(self, user_artist_matrix):
        self.user_artist_matrix = user_artist_matrix
        self.user_similarity = self._calculate_user_similarity()
        
    def _calculate_user_similarity(self):
        """Calculate user-user similarity using cosine similarity"""
        # Replace 0s with NaN for better similarity calculation
        matrix_for_sim = self.user_artist_matrix.replace(0, np.nan)
        
        # Fill NaN with user mean for similarity calculation
        user_means = matrix_for_sim.mean(axis=1)
        matrix_filled = matrix_for_sim.sub(user_means, axis=0).fillna(0)
        
        # Calculate cosine similarity
        user_similarity = cosine_similarity(matrix_filled)
        return pd.DataFrame(user_similarity, 
                          index=self.user_artist_matrix.index, 
                          columns=self.user_artist_matrix.index)
    
    def recommend(self, user_id, n_recommendations=5):
        """Recommend artists using user-based collaborative filtering"""
        if user_id not in self.user_artist_matrix.index:
            return f"User '{user_id}' not found"
        
        # Get user's ratings
        user_ratings = self.user_artist_matrix.loc[user_id]
        
        # Get similar users
        similar_users = self.user_similarity.loc[user_id].sort_values(ascending=False)[1:11]  # Top 10 similar users
        
        # Get recommendations based on similar users
        recommendations = {}
        
        for similar_user, similarity_score in similar_users.items():
            similar_user_ratings = self.user_artist_matrix.loc[similar_user]
            
            # Find artists that similar user liked but current user hasn't rated
            for artist, rating in similar_user_ratings.items():
                if user_ratings[artist] == 0 and rating > 3:  # Unrated by user, liked by similar user
                    if artist not in recommendations:
                        recommendations[artist] = 0
                    recommendations[artist] += similarity_score * rating
        
        # Sort recommendations
        sorted_recommendations = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)
        
        return sorted_recommendations[:n_recommendations]
    
    def get_user_top_artists(self, user_id, n=5):
        """Get user's top rated artists"""
        if user_id not in self.user_artist_matrix.index:
            return f"User '{user_id}' not found"
        
        user_ratings = self.user_artist_matrix.loc[user_id]
        top_artists = user_ratings.sort_values(ascending=False).head(n)
        return [(artist, rating) for artist, rating in top_artists.items() if rating > 0]

In [ ]:
# Initialize and test Collaborative Filtering Recommender
collab_recommender = CollaborativeFilteringRecommender(user_artist_matrix)

# Test with a few users
test_users = ['User_1', 'User_5', 'User_10']

print("👥 COLLABORATIVE FILTERING RECOMMENDATIONS")
print("=" * 60)

for user in test_users:
    print(f"\n👤 User: {user}")
    
    # Show user's top rated artists
    top_artists = collab_recommender.get_user_top_artists(user, n=3)
    print("Top rated artists:")
    for artist, rating in top_artists:
        print(f"  • {artist}: {rating}/5")
    
    # Get recommendations
    print("\nRecommendations:")
    recommendations = collab_recommender.recommend(user, n_recommendations=3)
    
    if isinstance(recommendations, str):
        print(recommendations)
    else:
        for i, (artist, score) in enumerate(recommendations, 1):
            print(f"{i}. {artist} (Score: {score:.3f})")
    
    print("-" * 50)

---
## 4. Matrix Factorization (Advanced Collaborative Filtering)

In [ ]:
class MatrixFactorizationRecommender:
    def __init__(self, user_artist_matrix, n_components=10):
        self.user_artist_matrix = user_artist_matrix
        self.n_components = n_components
        self.model = None
        self.W = None  # User factors
        self.H = None  # Artist factors
        
    def fit(self):
        """Fit the NMF model"""
        # Use Non-negative Matrix Factorization
        self.model = NMF(n_components=self.n_components, random_state=42, max_iter=500)
        
        # Fit the model
        self.W = self.model.fit_transform(self.user_artist_matrix)
        self.H = self.model.components_
        
        # Reconstruct the matrix
        self.predicted_ratings = np.dot(self.W, self.H)
        
    def recommend(self, user_id, n_recommendations=5):
        """Recommend artists using matrix factorization"""
        if user_id not in self.user_artist_matrix.index:
            return f"User '{user_id}' not found"
        
        user_idx = self.user_artist_matrix.index.get_loc(user_id)
        user_ratings = self.user_artist_matrix.loc[user_id]
        predicted_ratings = self.predicted_ratings[user_idx]
        
        # Get recommendations for unrated artists
        recommendations = []
        for i, (artist, actual_rating) in enumerate(user_ratings.items()):
            if actual_rating == 0:  # Unrated artist
                predicted_rating = predicted_ratings[i]
                recommendations.append((artist, predicted_rating))
        
        # Sort by predicted rating
        recommendations.sort(key=lambda x: x[1], reverse=True)
        
        return recommendations[:n_recommendations]
    
    def evaluate(self):
        """Evaluate the model using RMSE on non-zero entries"""
        mask = self.user_artist_matrix.values != 0
        actual = self.user_artist_matrix.values[mask]
        predicted = self.predicted_ratings[mask]
        
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        return rmse

In [ ]:
# Initialize and train Matrix Factorization Recommender
mf_recommender = MatrixFactorizationRecommender(user_artist_matrix, n_components=15)
mf_recommender.fit()

# Evaluate the model
rmse = mf_recommender.evaluate()
print(f"Matrix Factorization RMSE: {rmse:.3f}")

# Test with same users
print("\n🔄 MATRIX FACTORIZATION RECOMMENDATIONS")
print("=" * 60)

for user in test_users:
    print(f"\n👤 User: {user}")
    
    # Get recommendations
    print("Recommendations:")
    recommendations = mf_recommender.recommend(user, n_recommendations=3)
    
    if isinstance(recommendations, str):
        print(recommendations)
    else:
        for i, (artist, score) in enumerate(recommendations, 1):
            print(f"{i}. {artist} (Predicted Rating: {score:.3f})")
    
    print("-" * 50)

---
## 5. Hybrid Recommendation System

Combines content-based and collaborative filtering approaches.

In [ ]:
class HybridRecommender:
    def __init__(self, content_recommender, collab_recommender, content_weight=0.6):
        self.content_recommender = content_recommender
        self.collab_recommender = collab_recommender
        self.content_weight = content_weight
        self.collab_weight = 1 - content_weight
        
    def recommend(self, user_id, artist_preference=None, n_recommendations=5):
        """Hybrid recommendations combining content and collaborative filtering"""
        recommendations = {}
        
        # Get collaborative filtering recommendations
        collab_recs = self.collab_recommender.recommend(user_id, n_recommendations=10)
        if not isinstance(collab_recs, str):
            for artist, score in collab_recs:
                recommendations[artist] = self.collab_weight * score
        
        # Get content-based recommendations if user has an artist preference
        if artist_preference:
            content_recs = self.content_recommender.recommend(artist_preference, n_recommendations=10)
            if not isinstance(content_recs, str):
                for rec in content_recs:
                    artist = rec['artist']
                    score = rec['similarity_score']
                    
                    if artist in recommendations:
                        recommendations[artist] += self.content_weight * score
                    else:
                        recommendations[artist] = self.content_weight * score
        
        # Sort recommendations
        sorted_recommendations = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)
        
        return sorted_recommendations[:n_recommendations]
    
    def explain_recommendation(self, user_id, artist_preference=None):
        """Provide explanation for recommendations"""
        explanation = {
            'user_id': user_id,
            'artist_preference': artist_preference,
            'content_weight': self.content_weight,
            'collab_weight': self.collab_weight
        }
        
        # Get user's top artists
        if user_id in self.collab_recommender.user_artist_matrix.index:
            top_artists = self.collab_recommender.get_user_top_artists(user_id, n=3)
            explanation['user_top_artists'] = top_artists
        
        # Get artist genres if preference is given
        if artist_preference:
            genres = self.content_recommender.get_artist_genres(artist_preference)
            explanation['preferred_artist_genres'] = genres
        
        return explanation

In [ ]:
# Initialize Hybrid Recommender
hybrid_recommender = HybridRecommender(content_recommender, collab_recommender, content_weight=0.6)

print("🔄 HYBRID RECOMMENDATION SYSTEM")
print("=" * 60)

# Test hybrid recommendations
test_scenarios = [
    ('User_1', 'Taylor Swift'),
    ('User_5', 'The Beatles'),
    ('User_10', None)  # No content preference, pure collaborative
]

for user, artist_pref in test_scenarios:
    print(f"\n👤 User: {user}")
    if artist_pref:
        print(f"🎵 Artist Preference: {artist_pref}")
    
    # Get explanation
    explanation = hybrid_recommender.explain_recommendation(user, artist_pref)
    
    print("\n📊 User Profile:")
    if 'user_top_artists' in explanation:
        print("Top rated artists:")
        for artist, rating in explanation['user_top_artists']:
            print(f"  • {artist}: {rating}/5")
    
    if artist_pref:
        print(f"\n🎭 Preferred Artist Genres: {explanation['preferred_artist_genres']}")
    
    print(f"\n⚖️ Weights: Content {explanation['content_weight']:.1f} | Collaborative {explanation['collab_weight']:.1f}")
    
    # Get hybrid recommendations
    recommendations = hybrid_recommender.recommend(user, artist_pref, n_recommendations=5)
    
    print("\n🎯 Hybrid Recommendations:")
    for i, (artist, score) in enumerate(recommendations, 1):
        print(f"{i}. {artist} (Hybrid Score: {score:.3f})")
    
    print("-" * 60)

---
## 6. Performance Comparison and Analysis

In [ ]:
# Compare all recommendation systems
def compare_recommendations(user_id, artist_preference=None):
    """Compare recommendations from all systems"""
    
    results = {
        'User': user_id,
        'Artist Preference': artist_preference or 'None'
    }
    
    # Content-based (if artist preference given)
    if artist_preference:
        content_recs = content_recommender.recommend(artist_preference, n_recommendations=3)
        if not isinstance(content_recs, str):
            results['Content-Based'] = [rec['artist'] for rec in content_recs]
        else:
            results['Content-Based'] = ['N/A']
    else:
        results['Content-Based'] = ['N/A - No preference']
    
    # Collaborative filtering
    collab_recs = collab_recommender.recommend(user_id, n_recommendations=3)
    if not isinstance(collab_recs, str):
        results['Collaborative'] = [artist for artist, score in collab_recs]
    else:
        results['Collaborative'] = ['N/A']
    
    # Matrix factorization
    mf_recs = mf_recommender.recommend(user_id, n_recommendations=3)
    if not isinstance(mf_recs, str):
        results['Matrix Factorization'] = [artist for artist, score in mf_recs]
    else:
        results['Matrix Factorization'] = ['N/A']
    
    # Hybrid
    hybrid_recs = hybrid_recommender.recommend(user_id, artist_preference, n_recommendations=3)
    results['Hybrid'] = [artist for artist, score in hybrid_recs]
    
    return results

# Compare for test scenarios
print("📊 RECOMMENDATION SYSTEMS COMPARISON")
print("=" * 80)

comparison_results = []
for user, artist_pref in test_scenarios:
    result = compare_recommendations(user, artist_pref)
    comparison_results.append(result)
    
    print(f"\n👤 User: {user} | Artist Preference: {artist_pref or 'None'}")
    print("-" * 60)
    
    for method, recommendations in result.items():
        if method not in ['User', 'Artist Preference']:
            if isinstance(recommendations, list) and len(recommendations) > 0:
                recs_str = ', '.join(recommendations[:3])
            else:
                recs_str = 'No recommendations'
            print(f"{method:20}: {recs_str}")
    
    print("-" * 60)

In [ ]:
# Analysis of recommendation diversity
def analyze_diversity():
    """Analyze the diversity of recommendations across systems"""
    
    all_recommendations = {
        'Content-Based': set(),
        'Collaborative': set(),
        'Matrix Factorization': set(),
        'Hybrid': set()
    }
    
    # Collect all recommendations
    for user in ['User_1', 'User_5', 'User_10', 'User_15', 'User_20']:
        
        # Collaborative
        collab_recs = collab_recommender.recommend(user, n_recommendations=5)
        if not isinstance(collab_recs, str):
            all_recommendations['Collaborative'].update([artist for artist, score in collab_recs])
        
        # Matrix Factorization
        mf_recs = mf_recommender.recommend(user, n_recommendations=5)
        if not isinstance(mf_recs, str):
            all_recommendations['Matrix Factorization'].update([artist for artist, score in mf_recs])
        
        # Hybrid (without content preference)
        hybrid_recs = hybrid_recommender.recommend(user, n_recommendations=5)
        all_recommendations['Hybrid'].update([artist for artist, score in hybrid_recs])
    
    # Content-based (test with different artist preferences)
    for artist_pref in ['Taylor Swift', 'The Beatles', 'Drake', 'Miles Davis']:
        content_recs = content_recommender.recommend(artist_pref, n_recommendations=5)
        if not isinstance(content_recs, str):
            all_recommendations['Content-Based'].update([rec['artist'] for rec in content_recs])
    
    # Calculate diversity metrics
    print("🎯 RECOMMENDATION DIVERSITY ANALYSIS")
    print("=" * 50)
    
    for method, artists in all_recommendations.items():
        diversity = len(artists)
        coverage = len(artists) / len(unique_artists) * 100
        print(f"{method:20}: {diversity:2d} unique artists ({coverage:5.1f}% coverage)")
    
    return all_recommendations

diversity_results = analyze_diversity()

In [ ]:
# Visualize comparison results
plt.figure(figsize=(15, 10))

# Diversity comparison
plt.subplot(2, 2, 1)
methods = list(diversity_results.keys())
diversity_counts = [len(artists) for artists in diversity_results.values()]
coverage_pcts = [len(artists) / len(unique_artists) * 100 for artists in diversity_results.values()]

bars = plt.bar(methods, diversity_counts, color=['skyblue', 'lightgreen', 'orange', 'pink'])
plt.title('Recommendation Diversity\n(Unique Artists Recommended)')
plt.ylabel('Number of Unique Artists')
plt.xticks(rotation=45)
for bar, pct in zip(bars, coverage_pcts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{pct:.1f}%', ha='center', va='bottom')

# Genre distribution in recommendations
plt.subplot(2, 2, 2)
# Get genres for recommended artists (using collaborative as example)
collab_artists = list(diversity_results['Collaborative'])
collab_genres = music_df[music_df['name'].isin(collab_artists)]['genre_list'].value_counts().head(8)
collab_genres.plot(kind='pie', autopct='%1.1f%%')
plt.title('Genre Distribution\n(Collaborative Filtering)')
plt.ylabel('')

# Rating distribution
plt.subplot(2, 2, 3)
ratings_df['rating'].hist(bins=5, edgecolor='black', alpha=0.7)
plt.title('Distribution of User Ratings')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.xticks(range(1, 6))

# System complexity comparison
plt.subplot(2, 2, 4)
complexity_scores = {
    'Content-Based': 2,
    'Collaborative': 3,
    'Matrix Factorization': 4,
    'Hybrid': 5
}

plt.bar(complexity_scores.keys(), complexity_scores.values(), 
        color=['skyblue', 'lightgreen', 'orange', 'pink'])
plt.title('System Complexity\n(1=Simple, 5=Complex)')
plt.ylabel('Complexity Score')
plt.xticks(rotation=45)
plt.ylim(0, 6)

plt.tight_layout()
plt.show()

---
## 7. Summary and Conclusions

In [ ]:
print("🎯 RECOMMENDATION SYSTEMS COMPARISON SUMMARY")
print("=" * 70)

summary = """
📊 SYSTEM CHARACTERISTICS:

🎵 CONTENT-BASED FILTERING:
   ✅ Pros:
      • No cold start problem for new users
      • Transparent recommendations (based on genres)
      • Works well with item features
      • Doesn't need user rating data
   ❌ Cons:
      • Limited diversity (recommends similar genres)
      • Can't discover new genres user might like
      • Relies heavily on feature quality

👥 COLLABORATIVE FILTERING:
   ✅ Pros:
      • Can recommend across different genres
      • Leverages community preferences
      • Can discover new interests
   ❌ Cons:
      • Cold start problem for new users/items
      • Requires sufficient rating data
      • Sparsity issues

🔄 MATRIX FACTORIZATION:
   ✅ Pros:
      • Handles sparsity better than basic collaborative filtering
      • Captures latent factors
      • More accurate predictions
   ❌ Cons:
      • Less interpretable
      • Requires tuning of parameters
      • Still has cold start issues

🔗 HYBRID SYSTEM:
   ✅ Pros:
      • Combines strengths of both approaches
      • Better coverage and diversity
      • More robust recommendations
   ❌ Cons:
      • More complex to implement
      • Requires careful weight tuning
      • Higher computational cost

🎯 RECOMMENDATIONS FOR REAL-WORLD USE:

1. **New Platform/Cold Start**: Start with Content-Based
2. **Growing User Base**: Add Collaborative Filtering
3. **Mature Platform**: Implement Hybrid System
4. **Large Scale**: Consider Matrix Factorization or Deep Learning

📈 KEY METRICS TO MONITOR:
• Recommendation Accuracy (RMSE, MAE)
• Diversity and Coverage
• User Engagement (Click-through rates)
• Novelty and Serendipity
• System Performance and Scalability
"""

print(summary)

In [ ]:
# Final performance comparison table
performance_comparison = pd.DataFrame({
    'System': ['Content-Based', 'Collaborative', 'Matrix Factorization', 'Hybrid'],
    'Diversity (Unique Artists)': [len(diversity_results[method]) for method in 
                                  ['Content-Based', 'Collaborative', 'Matrix Factorization', 'Hybrid']],
    'Coverage (%)': [len(diversity_results[method]) / len(unique_artists) * 100 for method in 
                    ['Content-Based', 'Collaborative', 'Matrix Factorization', 'Hybrid']],
    'Complexity': [2, 3, 4, 5],
    'Cold Start Handling': ['Excellent', 'Poor', 'Poor', 'Good'],
    'Interpretability': ['High', 'Medium', 'Low', 'Medium'],
    'Best Use Case': ['New users', 'Established users', 'Large datasets', 'All scenarios']
})

print("\n📊 FINAL PERFORMANCE COMPARISON")
print("=" * 80)
print(performance_comparison.to_string(index=False))

print("\n\n🎉 Analysis Complete!")
print("This notebook demonstrated the implementation and comparison of different")
print("recommendation systems using the music dataset. Each approach has its")
print("strengths and is suitable for different scenarios and business requirements.")